# MyCON — اجرای واحد، کم‌ریسک و قابل‌استناد پایان‌نامه

**روش اصلی:** COLMAP + gsplat رسمی v1.5.3 روی فیلم واقعی Recorder. **NVIDIA LongSplat محفوظ است** و به‌عنوان روش پژوهشی مقایسه‌ای باقی می‌ماند، نه مرحله‌ای که ناخواسته آموزش شود.

## ترتیب دقیق

1. **Runtime = CPU**: سلول «اتصال به GitHub و Drive»؛ بعد «بررسی ZIP بدون GPU»؛ بعد «بازسازی هندسه COLMAP با CPU». هیچ یک آموزش GPU ندارند.
2. **Runtime → Change runtime type → T4 GPU**: بعد از تغییر Runtime، **سلول اتصال به GitHub و Drive را مجدداً اجرا کن**. سلول‌های CPU را دوباره اجرا نکن.
3. سلول «پیش‌اجرای GPU» را اجرا کن. باید وضعیت دقیق `PREFLIGHT_COLMAP_AND_GSPLAT_CUDA_PASSED_NO_TRAINING` صادر شود.
4. فقط پس از موفقیت و تأیید بودجه، در سلول آموزش مقدار `TRAIN_CONFIRMATION` را برابر `RUN_7000_STEPS_T4` قرار بده. در غیر این‌صورت آموزش حتی با Run All **شروع نمی‌شود**.
5. سلول وضعیت را هر زمان خواستی اجرا کن؛ خواندنی است و آموزش راه نمی‌اندازد.

**حفاظت از داده:** Recorder ZIP اصلی، pose.csv، IMU و checkpointهای قدیمی هرگز حذف یا بازنویسی نمی‌شوند. هر آزمایش مسیر `ATTEMPT` مستقل دارد. خروجی PLY و PSNR خوب به معنی تأیید متریک BIM نیست.

**نکته:** مقدار `CAPTURE_ZIP` را اگر جست‌وجوی خودکار مسیر یکتا را پیدا نکرد، فقط به آدرس واقعی ZIP اصلی داخل Drive تغییر بده. فایل آزمایشی ۱۰ثانیه‌ای معتبر نیست.


## ۱. اتصال GitHub و Google Drive (روی CPU و بعد از تغییر به T4 یک بار دیگر)


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, shutil, json

# Only these two values may need changing.
CAPTURE_ZIP = ""  # Example: /content/drive/MyDrive/MyCon_R4/captures/ORIGINAL_RECORDER.zip
ATTEMPT = "mycon_real_video_v1_20261105"
MYCON_BRANCH = "feature/mycon-single-final-acceptance-20261105"
MYCON_REPO_URL = "https://github.com/sayyedalimrj/MyCon.git"

drive.mount("/content/drive")  # Mount; does not sign out.
repo = Path("/content/MyCON_FINAL_ACCEPTANCE")
if repo.exists():
    assert (repo/".git").exists(), "Existing directory is not a Git checkout."
    origin = subprocess.check_output(
        ["git","-C",str(repo),"remote","get-url","origin"],text=True).strip()
    assert origin in (MYCON_REPO_URL,
                      "git@github.com:sayyedalimrj/MyCon.git"), "Unexpected repo origin."
    dirty = subprocess.check_output(["git","-C",str(repo),"status","--porcelain"],text=True)
    assert not dirty.strip(), "Local repo has changes. Preserving it; do not overwrite."
    subprocess.run(["git","-C",str(repo),"fetch","origin",MYCON_BRANCH],check=True)
    subprocess.run(["git","-C",str(repo),"checkout",MYCON_BRANCH],check=True)
    subprocess.run(["git","-C",str(repo),"pull","--ff-only","origin",MYCON_BRANCH],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch",MYCON_BRANCH,
                    MYCON_REPO_URL,str(repo)],check=True)

commit = subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
script = repo/"repository/scripts/mycon_colmap_gsplat.py"
assert script.is_file(), "Incomplete checkout: official MyCON entrypoint missing."
print("MYCON GIT SHA:",commit)
print("SELECTED ATTEMPT:",ATTEMPT)
print("GPU training started: NO")

def run_mycon(mode, approve=False):
    cmd=[sys.executable,str(script),"--mode",mode,"--attempt",ATTEMPT]
    if CAPTURE_ZIP.strip():
        cmd+=["--capture",CAPTURE_ZIP.strip()]
    if approve:
        cmd+=["--approve-training"]
    print("MYCON START:",mode,"| approved GPU training:",approve,flush=True)
    return subprocess.run(cmd,check=True)


## ۲. بررسی اصالت و سلامت ZIP — فقط روی CPU، بدون pip و بدون GPU


In [ ]:
run_mycon("verify")

## ۳. ساخت هندسه COLMAP — فقط روی CPU

ابتدا Runtime را **CPU** نگه دار. اجرای این مرحله روی T4 عمداً مسدود شده تا اعتبار GPU برای COLMAP مصرف نشود. نتیجه هندسه و رسید آن در Google Drive ذخیره می‌شود.


In [ ]:
if shutil.which("nvidia-smi"):
    active=subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True,check=False)
    if active.returncode==0:
        raise RuntimeError("CPU_ONLY_STAGE: switch Colab runtime to CPU before geometry; do not waste T4 credits.")
required = [p for p in ("ffmpeg","ffprobe","colmap") if shutil.which(p) is None]
if required:
    print("Installing CPU-only OS packages needed for:",required,flush=True)
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","ffmpeg","colmap"],check=True)
run_mycon("geometry")
print("CPU COLMAP stage complete. NOW switch Runtime to T4 GPU, rerun setup cell, then GPU preflight.")


## ۴. پیش‌اجرای واقعی GPU — پس از تعویض Runtime به T4 و اجرای مجدد سلول اتصال

این مرحله آموزش ۷۰۰۰گامی انجام نمی‌دهد؛ صرفاً پکیج‌های ایزوله Python 3.10، Torch/CUDA، wheel رسمی gsplat، مدل دوربین COLMAP و کرنل‌های واقعی را بررسی می‌کند. **نصب پکیج روی T4 زمان GPU می‌گیرد**؛ مرحله CPU پیش‌تر از این مصرف جلوگیری کرده است.


In [ ]:
assert shutil.which("nvidia-smi"), "Select GPU in Runtime > Change runtime type."
gpu=subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap",
                    "--format=csv,noheader"],capture_output=True,text=True,check=True)
print("GPU:",gpu.stdout.strip())
assert "T4" in gpu.stdout and "7.5" in gpu.stdout, (
    "This frozen CUDA12.1/gsplat environment is certified for T4 SM7.5 only.")
run_mycon("preflight")
print("Preflight invoked. Read the exact PASS status in the next cell before any training.")


## ۵. آموزش واقعی ۷۰۰۰ مرحله‌ای — فقط با تأیید صریح

به‌صورت پیش‌فرض این سلول هیچ کاری نمی‌کند. صرفاً بعد از موفقیت preflight و تصمیم به مصرف GPU، متن تأیید را خودت تغییر بده.


In [ ]:
TRAIN_CONFIRMATION = ""  # To approve, enter exactly: RUN_7000_STEPS_T4
if TRAIN_CONFIRMATION == "RUN_7000_STEPS_T4":
    assert shutil.which("nvidia-smi"), "CUDA T4 is not active."
    run_mycon("all",approve=True)
else:
    print("GPU TRAINING BLOCKED BY DEFAULT. No training was started.")


## ۶. گزارش وضعیت — بدون شروع آموزش


In [ ]:
run_mycon("status")

## وضعیت شاخه NVIDIA و مرحله BIM

- کد رسمی [NVIDIA LongSplat](https://github.com/NVlabs/LongSplat) در همان PR محفوظ است. پیش از اجرای آن باید محیط PyTorch3D و MASt3R با کرنل‌های واقعی T4 تست شوند؛ LongSplat در این نوت‌بوک **خودکار اجرا نمی‌شود** تا به‌صورت ناخواسته اعتبار GPU مصرف نشود.
- فایل خروجی مستقل `bim_candidate/colmap_sparse_unscaled.ply` فقط «هندسه SfM با مقیاس نامعلوم» است؛ برای گزارش پیشرفت، ابتدا باید فایل IFC واقعی و نقاط کنترل مستقل به مرحله ثبت متریک داده شوند.
- جدول نسخه‌ها، SHA رسمی wheel و مشکلات شناسایی‌شده در `repository/docs/MYCON_RUNTIME_COMPATIBILITY_LOCK_20261105.json` است.
- اگر هر مرحله خطا داد، **نوت‌بوک دیگری یا دستور نصب تصادفی اجرا نکن**؛ خروجی همان مرحله و فایل `logs/pipeline.log` را بررسی کنیم. فایل‌های قبلی محفوظ‌اند.

[PR #31](https://github.com/sayyedalimrj/MyCon/pull/31) — [راهنمای فارسی](https://github.com/sayyedalimrj/MyCon/blob/feature/mycon-single-final-acceptance-20261105/repository/docs/MYCON_ONE_FINAL_RUN_FA.md)
